In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import numpy as np

# Example: create dummy predictions
# Replace `test.csv` with your actual test file if needed

sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
sample.to_csv("submission.csv",index=False)

In [3]:
df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [4]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

# Milestone - 2

## Q1 
Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text. What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51? Note: We follow zero-indexing here.

In [5]:
# 1. Import the required function
from datasets import load_dataset

# 2. Load the CSV file 
dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv", split="train")

# 3. Define the function to combine the columns
def combine_text(example):
    # Ensure both are treated as strings and combine with a space
    example["combined_text"] = str(example["prompt"]) + " " + str(example["A"])
    return example

# 4. Apply the function to the entire dataset
dataset = dataset.map(combine_text)

# 5. Access the row at index 51 (zero-indexed) and get the string length
target_string = dataset[51]["combined_text"]
char_length = len(target_string)

# Output the exact character length
print(f"The exact character length at index 51 is: {char_length}")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

The exact character length at index 51 is: 614


## Q2
Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer?  

In [6]:
from transformers import AutoTokenizer

# 1. Initialize the bert-base-uncased tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# 2. Access the vocabulary size property
vocab_size = tokenizer.vocab_size

# 3. Alternatively, get the length of the full vocabulary dictionary (includes added tokens if any)
total_vocab_len = len(tokenizer)

print(f"The exact tokenizer vocab_size property is: {vocab_size}")
print(f"The exact total length using len() is: {total_vocab_len}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

The exact tokenizer vocab_size property is: 30522
The exact total length using len() is: 30522


## Q3
Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token.

In [7]:
# 2. Method A: Use the dedicated property
sep_id_property = tokenizer.sep_token_id

# 3. Method B: Convert the token string directly
sep_id_method = tokenizer.convert_tokens_to_ids("[SEP]")

print(f"The exact integer ID for [SEP] via property is: {sep_id_property}")
print(f"The exact integer ID for [SEP] via conversion is: {sep_id_method}")

The exact integer ID for [SEP] via property is: 102
The exact integer ID for [SEP] via conversion is: 102


## Q4
Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors). 

What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [8]:
# 1. Clean the column: cast every prompt to a string and handle potential None/NaN values
# This ensures a clean list[str] is fed to the tokenizer
cleaned_prompts = [str(p) if p is not None else "" for p in dataset["prompt"]]

# 3. Tokenize the clean python list simultaneously
encoded_inputs = tokenizer(
    cleaned_prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# 4. Extract the input_ids tensor and print its shape
tensor_shape = encoded_inputs["input_ids"].shape

print(f"The exact geometric shape of the input_ids tensor is: {tensor_shape}")

The exact geometric shape of the input_ids tensor is: torch.Size([2000, 128])


## Q5
BERT/RoBERTa Architecture & Attention Mechanisms

A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer. 

In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

answer = 64

## Q6
Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object. 

What is the exact shape of the last_hidden_state tensor returned? 

Note: We follow zero-indexing here.

In [9]:
import torch
from transformers import  AutoModel

model = AutoModel.from_pretrained("bert-base-uncased")

# 2. Load the dataset and extract the prompt at index 0
prompt_text = str(dataset[0]["prompt"])

# 3. Tokenize with default settings (no padding/truncation)
inputs = tokenizer(prompt_text, return_tensors="pt")

# 4. Pass the tokenized inputs through the model
with torch.no_grad():
    outputs = model(**inputs)

# 5. Extract the last_hidden_state and get its shape
hidden_shape = outputs.last_hidden_state.shape

print(f"The exact shape of the last_hidden_state tensor is: {hidden_shape}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


The exact shape of the last_hidden_state tensor is: torch.Size([1, 31, 768])


## Q7
Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).  

In [10]:

prompt_text = str(dataset[0]["prompt"])
inputs = tokenizer(prompt_text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

# 2. Extract the first 5 values of the [CLS] token vector
# last_hidden_state shape: [batch_size, sequence_length, hidden_size]
cls_vector_first_five = outputs.last_hidden_state[0, 0, :5]

# 3. Sum the values and convert to a standard Python float
vector_sum = torch.sum(cls_vector_first_five).item()

# 4. Print the result rounded to 4 decimal places
print(f"The first 5 values are: {cls_vector_first_five.tolist()}")
print(f"The exact sum rounded to 4 decimal places is: {round(vector_sum, 4)}")

The first 5 values are: [-0.46766412258148193, -0.07544441521167755, -0.20190182328224182, -0.007064236328005791, -0.4480222463607788]
The exact sum rounded to 4 decimal places is: -1.2001


## Q8
Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0). 

What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).  

In [11]:
model = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)

text = "Light-ion fusion is a technique."

# 2. Tokenize the text
inputs = tokenizer(text, return_tensors="pt")

# Convert IDs back to tokens so we can visually confirm the index of "fusion"
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
fusion_index = tokens.index("fusion")

print(f"Tokens list: {tokens}")
print(f"The exact token index for 'fusion' is: {fusion_index}")

# 3. Pass through the model
with torch.no_grad():
    outputs = model(**inputs)

# 4. Extract attention matrix: Last layer [-1], First head [0]
# Shape: [batch_size, num_heads, seq_len, seq_len]
attention_matrix = outputs.attentions[-1][0, 0]

# 5. Get the specific weight from [CLS] (index 0) to 'fusion'
attention_weight = attention_matrix[0, fusion_index].item()

print(f"The exact attention weight rounded to 4 decimal places is: {round(attention_weight, 4)}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens list: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
The exact token index for 'fusion' is: 4
The exact attention weight rounded to 4 decimal places is: 0.1025


## Q9
Context-Aware Embeddings 

Initialize the sentence-transformers/all-MiniLM-L6-v2 model. Use the model's .encode() method to generate embeddings for both the prompt and Option B for row ID 0. Calculate the cosine similarity between these two vectors specifically using the sentence_transformers.util.cos_sim() function. What is the resulting similarity score rounded to 4 decimal places? Note: We follow zero-indexing here.

In [12]:
from sentence_transformers import SentenceTransformer, util

# 1. Initialize the SentenceTransformer model
# This will download the model weights if not already cached
model = SentenceTransformer("all-MiniLM-L6-v2")


# 2. Extract the text for the prompt and Option B at index 0
prompt_text = str(dataset[0]["prompt"])
option_b_text = str(dataset[0]["B"])

# 3. Generate the sentence embeddings
prompt_embedding = model.encode(prompt_text)
option_b_embedding = model.encode(option_b_text)

# 4. Calculate cosine similarity
similarity_tensor = util.cos_sim(prompt_embedding, option_b_embedding)

# 5. Extract the single float value from the tensor and round it
similarity_score = similarity_tensor.item()

print(f"The exact cosine similarity score rounded to 4 decimal places is: {round(similarity_score, 4)}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

The exact cosine similarity score rounded to 4 decimal places is: 0.7658


## Q10
Build two complete ranking pipelines evaluating every row in train.csv.

Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.

Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.

First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set? 

Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?  

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings

# Suppress standard sklearn warnings for clean output
warnings.filterwarnings("ignore")

# 1. Load the dataset and initialize the model
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Helper function to calculate MAP@3 for a single row
def get_map3_score(top_3_preds, true_answer):
    if true_answer == top_3_preds[0]: 
        return 1.0
    elif true_answer == top_3_preds[1]: 
        return 0.5
    elif true_answer == top_3_preds[2]: 
        return 1.0 / 3.0
    else: 
        return 0.0

options_labels = ['A', 'B', 'C', 'D', 'E']
minilm_map3_total = 0.0
improvement_count = 0

print("Evaluating 2000 rows. This may take a moment...")

# 3. Iterate through every question in the dataset
for row in dataset:
    # Safely extract and cast text to strings
    prompt = str(row['prompt'])
    ans_texts = [str(row[opt]) for opt in options_labels]
    true_answer = row['answer']
    
    # --- PIPELINE 1: TF-IDF ---
    tfidf_top_3 = []
    try:
        tfidf = TfidfVectorizer()
        # Fit TF-IDF on the prompt + the 5 options for this specific question
        tfidf_matrix = tfidf.fit_transform([prompt] + ans_texts)
        # Cosine similarity of prompt (index 0) vs options (index 1 to 5)
        tfidf_sims = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:]).flatten()
        # Sort descending and grab top 3 labels
        tfidf_top_indices = np.argsort(tfidf_sims)[::-1][:3]
        tfidf_top_3 = [options_labels[i] for i in tfidf_top_indices]
    except ValueError:
        # Fallback in case a row is completely empty/corrupted and TF-IDF fails
        tfidf_top_3 = ['A', 'B', 'C']

    # --- PIPELINE 2: MINILM ---
    # Encode the prompt + the 5 options into dense vectors
    embeddings = model.encode([prompt] + ans_texts, convert_to_tensor=True)
    # Cosine similarity of prompt (index 0) vs options (index 1 to 5)
    minilm_sims = util.cos_sim(embeddings[0], embeddings[1:]).flatten().cpu().numpy()
    # Sort descending and grab top 3 labels
    minilm_top_indices = np.argsort(minilm_sims)[::-1][:3]
    minilm_top_3 = [options_labels[i] for i in minilm_top_indices]

    # --- EVALUATION ---
    # Add to running MAP@3 total
    minilm_map3_total += get_map3_score(minilm_top_3, true_answer)
    
    # Check the specific improvement condition
    tfidf_has_ans = true_answer in tfidf_top_3
    minilm_has_ans = true_answer in minilm_top_3
    
    if (not tfidf_has_ans) and minilm_has_ans:
        improvement_count += 1

# 4. Final Calculations
total_questions = len(dataset)
final_minilm_map3 = minilm_map3_total / total_questions

print("\n--- RESULTS ---")
print(f"Final MAP@3 score for MiniLM: {round(final_minilm_map3, 4)}")
print(f"Questions where MiniLM succeeded but TF-IDF failed: {improvement_count}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating 2000 rows. This may take a moment...

--- RESULTS ---
Final MAP@3 score for MiniLM: 0.4231
Questions where MiniLM succeeded but TF-IDF failed: 564


## Q11
Zero-shot classification concepts 

Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [14]:
from transformers import pipeline

# 1. Initialize the zero-shot classification pipeline
# This downloads facebook/bart-large-mnli by default
classifier = pipeline("zero-shot-classification",model="facebook/bart-large-mnli")

# 2. Extract the prompt and options A, B, C from row ID 1 (the 2nd row)
row = dataset[1]
prompt_text = str(row["prompt"])
candidate_options = [str(row["A"]), str(row["B"]), str(row["C"])]

# 3. Run the zero-shot classifier
outputs = classifier(
    prompt_text,
    candidate_labels=candidate_options,
    multi_label=False # We want a mutually exclusive probability distribution
)

# 4. The pipeline automatically sorts the results from highest to lowest score
top_label = outputs["labels"][0]
top_score = outputs["scores"][0]

print(f"Prompt text: {prompt_text}\n")
print(f"Top-ranked Option: {top_label}")
print(f"The exact probability score rounded to 4 decimal places is: {round(top_score, 4)}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Prompt text: What is accelerator-based light-ion fusion?

Top-ranked Option: Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.
The exact probability score rounded to 4 decimal places is: 0.4575


## Q12
Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True. 

What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [15]:
from transformers import pipeline


# 1. Extract row index 1 data
row = dataset[1]
prompt_text = str(row["prompt"])
candidate_options = [str(row["A"]), str(row["B"]), str(row["C"])]

# 2. Run multi_label=False (Softmax)
outputs_softmax = classifier(
    prompt_text,
    candidate_labels=candidate_options,
    multi_label=False
)
sum_softmax = sum(outputs_softmax["scores"])

# 3. Run multi_label=True (Independent Sigmoids)
outputs_sigmoid = classifier(
    prompt_text,
    candidate_labels=candidate_options,
    multi_label=True
)
sum_sigmoid = sum(outputs_sigmoid["scores"])

# 4. Calculate the absolute difference
abs_diff = abs(sum_softmax - sum_sigmoid)

print(f"Sum of probabilities (multi_label=False): {sum_softmax:.6f}")
print(f"Sum of probabilities (multi_label=True): {sum_sigmoid:.6f}")
print(f"The absolute difference rounded to 4 decimal places is: {round(abs_diff, 4)}")

Sum of probabilities (multi_label=False): 1.000000
Sum of probabilities (multi_label=True): 0.000510
The absolute difference rounded to 4 decimal places is: 0.9995


## Q13
Let's try Generative AI instead of Classification. 

Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." 
Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model? 

In [16]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load the tokenizer and Seq2Seq model explicitly
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

# 2. Extract row ID 0 (the 1st row) from your dataset
row = dataset[0]
prompt_text = str(row["prompt"])
option_a = str(row["A"])
option_b = str(row["B"])

# 3. Construct the exact prompt string specified by the milestone
formatted_input = f"Question: {prompt_text}. Is the correct answer A: {option_a} or B: {option_b}? Answer with just the letter A or B."

# 4. Tokenize the input string
inputs = tokenizer(formatted_input, return_tensors="pt")

# 5. Generate the output tokens directly using the model
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=5)

# 6. Decode the tokens back into an exact text string
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Constructed Input:\n{formatted_input}\n")
print(f"The exact string output returned by the model is: '{generated_text}'")

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Constructed Input:
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.. Is the correct answer A: Martin Heidegger believes that humans exist within a time continuum that is infinite and does not have a defined beginning or end. The relationship to the past involves acknowledging it as a historical era, and the relationship to the future involves creating a world that will endure beyond one's own time. or B: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.? Answer with just the letter A or B.

The exact string output returned by the model is: 'B'


# Milestone -3

Q1 . Milestone 3 setup (run first, before answering any questions)
Run this code in a code cell
 before answering any of the question. This will create your knowledge 
base and the FAISS index for this milestone.

In [17]:
!pip install faiss-cpu #install FAISS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 86.3 MB/s eta 0:00:00


In [18]:
import faiss 
from sentence_transformers import SentenceTransformer, CrossEncoder 
from transformers import AutoTokenizer, pipeline 
from sklearn.feature_extraction.text import TfidfVectorizer 
from sklearn.metrics.pairwise import cosine_similarity 

train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv') 

print("Creating knowledge base")
kb = [] 
for idx, row in train.iterrows(): 
    correct_letter = row['answer'] 
    kb.append(str(row[correct_letter])) 

print("Loading embedding model and creating index") 
model = SentenceTransformer('all-MiniLM-L6-v2') 
kb_embeddings = model.encode(kb, show_progress_bar=False) 
index = faiss.IndexFlatL2(kb_embeddings.shape[1]) 
index.add(kb_embeddings)

print("Knowledge base successfully created")

Creating knowledge base
Loading embedding model and creating index


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Knowledge base successfully created


In [19]:

zs = pipeline("zero-shot-classification", model="facebook/bart-large-mnli") 
row_150 = train.iloc[150] 
prompt_150 = str(row_150['prompt']) 
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']), str(row_150['D']), str(row_150['E'])] 
ans_150 = str(row_150[row_150['answer']])


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Q1. Run the zero-shot classifier on facebook/bart-large-mnli on prompt for the row index 150. Pass the 5 options (A-E) candidate_labels. What is the predicted probability score assigned to the ground-truth correct option (option in the answer column)? (Round to 3 decimal points)


In [20]:
result = zs(prompt_150, candidate_labels=labels_150)

# 2. Find where the correct answer is located in the results dictionary
correct_idx = result['labels'].index(ans_150)

# 3. Extract the matching score and round it to 3 decimal places
correct_score = result['scores'][correct_idx]
final_answer = round(correct_score, 3)

# 4. Display the result
print(f"The predicted probability score for the correct option is: {final_answer}")

The predicted probability score for the correct option is: 0.384


Q2. Embed the prompt for row index 150 using all-MiniLM-L6-v2. Query your FAISS index to retrieve the top k=10 most similar documents. At what exact rank (1 through 10) did FAISS place the true correct document (which is the document originally located at index 150 in the KB)?

In [21]:
# 1. Embed the prompt for row index 150
# Note: FAISS requires the embedding to be a 2D numpy array, so we pass it as a list
prompt_embedding = model.encode([prompt_150], show_progress_bar=False)

# 2. Query the FAISS index for the top k=10 most similar documents
k = 10
distances, indices = index.search(prompt_embedding, k)

# 3. Extract the top 10 retrieved indices
# indices[0] contains the matched indices from the knowledge base for our single prompt
top_10_indices = indices[0].tolist()

# 4. Find the exact rank of document index 150
target_index = 150

if target_index in top_10_indices:
    # We add 1 because Python lists are 0-indexed, but the question asks for a rank from 1 to 10
    rank = top_10_indices.index(target_index) + 1
    print(f"The true correct document (index 150) was placed at rank: {rank}")
else:
    print("The true correct document is NOT in the top 10 retrieved documents.")

The true correct document (index 150) was placed at rank: 10


Q3. Take the top 10 documents retrieved by FAISS in the previous question. Load cross-encoder/ms-marco-MiniLM-L-6-v2. Score the prompt against these 10 documents and sort them by the cross-encoder's score. At what exact rank (1 through 10) does the Cross-Encoder place the true correct document?

In [22]:
# 1. Load the Cross-Encoder model
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# 2. Fetch the actual text for the top 10 documents using the indices from Q2
# (Make sure top_10_indices is still stored in memory from your previous cell)
docs_10 = [kb[i] for i in top_10_indices]

# 3. Create the prompt-context pairs for the Cross-Encoder
pairs = [[prompt_150, doc] for doc in docs_10]

# 4. Get the relevance scores for each pair
ce_scores = cross_encoder.predict(pairs)

# 5. Tie the new scores back to their original knowledge base indices
# We use zip() to match each index with its new score (e.g., (1045, 8.43), (150, 9.99))
scored_indices = list(zip(top_10_indices, ce_scores))

# 6. Sort the pairs based on the score (the second item in the pair: x[1]) in descending order
sorted_scored_indices = sorted(scored_indices, key=lambda x: x[1], reverse=True)

# 7. Extract just the sorted indices into a fresh list
final_sorted_indices = [index for index, score in sorted_scored_indices]

# 8. Find the exact rank of our target document
target_index = 150
new_rank = final_sorted_indices.index(target_index) + 1

print(f"The Cross-Encoder placed the true correct document at rank: {new_rank}")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

The Cross-Encoder placed the true correct document at rank: 1


Q4. Retrieve the top k=5 documents for the prompt at row index 42. Concatenate them with a single space between each. Create a string: "Context: [concatenated_docs] Question: [prompt]". Tokenize this string using the bert-base-uncased tokenizer (without truncation). Exactly how many total tokens does this generate?


In [23]:
# 1. Load the specific BERT tokenizer requested
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# 2. Extract the prompt for row index 42
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

# 3. Embed the prompt and search FAISS for top k=5
prompt_42_embedding = model.encode([prompt_42], show_progress_bar=False)
k = 5
distances, indices = index.search(prompt_42_embedding, k)

# 4. Retrieve and concatenate the top 5 documents
top_5_indices = indices[0].tolist()
retrieved_docs = [kb[i] for i in top_5_indices]

# Join them together with a single space
concatenated_docs = " ".join(retrieved_docs) 

# 5. Construct the final string exactly as requested
final_string = f"Context: {concatenated_docs} Question: {prompt_42}"

# 6. Tokenize the string (without truncation) and count the tokens
# We access 'input_ids' because it contains the actual list of numerical tokens
tokens = tokenizer(final_string, truncation=False)
total_tokens = len(tokens['input_ids'])

print(f"The total number of tokens generated is: {total_tokens}")

The total number of tokens generated is: 216


Q5. Retrieve the exact true document for row index 150 from your KB. Create a RAG string: "Context: [true_document] Question: [prompt]". Run the same zero-shot classification from Question 1 on this augmented string. What is the new predicted probability score of the ground-truth correct option? (Round to 3 decimal places).
*


In [24]:
# 1. Retrieve the exact true document for row 150 from the knowledge base
true_document = kb[150]

# 2. Construct the RAG string perfectly formatted
rag_string = f"Context: {true_document} Question: {prompt_150}"

# 3. Run the zero-shot pipeline on this new augmented string
rag_result = zs(rag_string, candidate_labels=labels_150)

# 4. Find where the correct answer is located in the new results dictionary
rag_correct_idx = rag_result['labels'].index(ans_150)

# 5. Extract the matching score and round it to 3 decimal places
rag_correct_score = rag_result['scores'][rag_correct_idx]
final_rag_score = round(rag_correct_score, 3)

# 6. Display the result
print(f"The new predicted probability score with RAG context is: {final_rag_score}")

The new predicted probability score with RAG context is: 0.989


Q6. What happens if your vector database retrieves the wrong information? Take the prompt for row index 150. Manually force the context to be the document located at KB index 999 (a completely unrelated fact). Run the zero-shot classifier on this "Adversarial RAG" string. What is the probability of the correct option now? (Round to 3 decimal places).

In [25]:
# 1. Retrieve the WRONG document from index 999
wrong_document = kb[999]

# 2. Construct the Adversarial RAG string
adv_rag_string = f"Context: {wrong_document} Question: {prompt_150}"

# 3. Run the zero-shot pipeline on this misleading string
adv_result = zs(adv_rag_string, candidate_labels=labels_150)

# 4. Find where the correct answer is located in the new results dictionary
adv_correct_idx = adv_result['labels'].index(ans_150)

# 5. Extract the matching score and round it to 3 decimal places
adv_correct_score = adv_result['scores'][adv_correct_idx]
final_adv_score = round(adv_correct_score, 3)

# 6. Display the result
print(f"The probability score with adversarial context is: {final_adv_score}")

The probability score with adversarial context is: 0.529


In RAG, a "Hit" occurs if the retrieved context contains the facts needed to answer the question.

Q7. For the first 100 rows of train.csv (indices 0-99), retrieve the top k=5 documents for each prompt. If the exact string of the row's correct option is found inside any of those 5 retrieved documents, it counts as a hit. What is the exact Hit Rate percentage (0 to 100) for these 100 rows? (Round to 1 decimal place).


In [26]:
# 1. Initialize our hit counter and set parameters
hits = 0
k = 5
num_rows = 100

# 2. Loop through the first 100 rows of the training dataset
for i in range(num_rows):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    
    # 3. Get the exact text of the correct option
    correct_letter = row['answer']
    correct_text = str(row[correct_letter])
    
    # 4. Embed the prompt and search FAISS for top k=5
    prompt_embedding = model.encode([prompt], show_progress_bar=False)
    distances, indices = index.search(prompt_embedding, k)
    top_5_indices = indices[0].tolist()
    
    # 5. Check if the correct_text is inside ANY of the 5 retrieved documents
    hit_found = False
    for idx in top_5_indices:
        retrieved_doc = kb[idx]
        if correct_text in retrieved_doc:
            hit_found = True
            break # Stop looking if we found a hit in this set of 5
            
    # 6. If we found a hit, add it to our counter
    if hit_found:
        hits += 1

# 7. Calculate the percentage and round to 1 decimal place
hit_rate = (hits / num_rows) * 100
final_hit_rate = round(hit_rate, 1)

print(f"The exact Hit Rate for the first {num_rows} rows is: {final_hit_rate}%")

The exact Hit Rate for the first 100 rows is: 73.0%


Q8. Build a loop that processes the first 20 rows (indices 0 through 19) of train.csv.
For each row, your pipeline must do the following in order:

Retrieve: Embed the prompt and retrieve the top k=5 documents from your FAISS Knowledge Base.

Rerank: Pass the prompt and those 5 documents into the ms-marco-MiniLM-L-6-v2 Cross-Encoder. Select the single document with the highest cross-encoder score.

Augment: Create your RAG string exactly formatted as: "Context: [best_document] Question: [prompt]".

Predict: Pass this augmented string to the facebook/bart-large-mnli zero-shot classifier, using the 5 options (A, B, C, D, E) as your candidate_labels.

Score: Look at the probability scores output by the model. Rank the options from highest probability to lowest. Take the top 3 letters (e.g., ['C', 'A', 'E']) and calculate the MAP@3 for that row.

What is the final average MAP@3 score of this state-of-the-art RAG pipeline across these 20 rows? (Round to 3 decimal places).

In [27]:
# 1. Initialize variables
num_rows = 20
total_map3_score = 0.0
k = 5

# 2. Loop through the first 20 rows of the training dataset
for i in range(num_rows):
    row = train.iloc[i]
    prompt = str(row['prompt'])
    correct_letter = row['answer']
    
    # Map the text options back to their letters so we can calculate MAP@3
    option_map = {
        str(row['A']): 'A',
        str(row['B']): 'B',
        str(row['C']): 'C',
        str(row['D']): 'D',
        str(row['E']): 'E'
    }
    candidate_labels = list(option_map.keys())
    
    # --- STAGE 1: RETRIEVAL ---
    prompt_embedding = model.encode([prompt], show_progress_bar=False)
    distances, indices = index.search(prompt_embedding, k)
    top_5_indices = indices[0].tolist()
    docs_5 = [kb[idx] for idx in top_5_indices]
    
    # --- STAGE 2: RERANKING ---
    pairs = [[prompt, doc] for doc in docs_5]
    ce_scores = cross_encoder.predict(pairs)
    
    # Find the index of the highest score using argmax
    best_doc_idx = ce_scores.argmax()
    best_document = docs_5[best_doc_idx]
    
    # --- STAGE 3: AUGMENT ---
    rag_string = f"Context: {best_document} Question: {prompt}"
    
    # --- STAGE 4: PREDICT ---
    result = zs(rag_string, candidate_labels=candidate_labels)
    
    # --- STAGE 5: SCORE (MAP@3) ---
    # The zero-shot pipeline automatically sorts 'labels' by highest probability
    # We grab the top 3 predicted text options
    top_3_text = result['labels'][:3]
    
    # Convert those text options back to 'A', 'B', 'C', 'D', 'E'
    top_3_letters = [option_map[text] for text in top_3_text]
    
    # Calculate MAP@3 for this row
    if correct_letter == top_3_letters[0]:
        total_map3_score += 1.0
    elif correct_letter == top_3_letters[1]:
        total_map3_score += 0.5
    elif correct_letter == top_3_letters[2]:
        total_map3_score += (1.0 / 3.0)

# 3. Calculate final average MAP@3 and round to 3 decimal places
average_map3 = total_map3_score / num_rows
final_score = round(average_map3, 3)

print(f"The final average MAP@3 score across the first {num_rows} rows is: {final_score}")

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


The final average MAP@3 score across the first 20 rows is: 0.975
